# Task 1: Telco Search
## Hybrid Search


Setup

In [1]:
pip install PyPDF2 rank_bm25 sentence-transformers chromadb pinecone-client pandas numpy python-dotenv langchain-text-splitters


   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 548.1/548.1 kB 4.6 MB/s  0:00:00

   ---------- -----------------------------  3/11 [PyPDF2]
   ---------- -----------------------------  3/11 [PyPDF2]
   -------------- -------------------------  4/11 [langchain-protocol]
   --------------------- ------------------  6/11 [requests-toolbelt]
   ----------------------------- ----------  8/11 [langsmith]
   ----------------------------- ----------  8/11 [langsmith]
   ----------------------------- ----------  8/11 [langsmith]
   ----------------------------- ----------  8/11 [langsmith]
   ----------------------------- ----------  8/11 [langsmith]
   -------------------------------- -------  9/11 [langchain-core]
   -------------------------------- -------  9/11 [langchain-core]
   -------------------------------- -------  9/11 [langchain-core]
   -------------------------------- -------  9/11 [langchain-core]
   -----------

Setup

In [ ]:
import os
import PyPDF2
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Keyword Search
from rank_bm25 import BM25Okapi

# Semantic Search
from sentence_transformers import SentenceTransformer

# Vector Database
import chromadb
from pinecone import Pinecone, ServerlessSpec

# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(override=True)

print("Semua library berhasil diimpor.")

Semua library berhasil diimpor.


Extract PDF

In [ ]:
def extract_text_from_pdf(pdf_path):
    try:
        reader = PyPDF2.PdfReader(pdf_path)
        pages_data = []
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text:
                # Membersihkan teks dari enter berlebih
                clean_text = " ".join(text.split())
                pages_data.append({
                    "text": clean_text,
                    "metadata": {"page": i + 1, "source": pdf_path}
                })
        print(f"Berhasil membaca {len(pages_data)} halaman dari {pdf_path}")
        return pages_data
    except FileNotFoundError:
        print(f"File {pdf_path} tidak ditemukan!")
        return []

pdf_file = "docs_telco_v2.pdf"
raw_pages = extract_text_from_pdf(pdf_file)

Berhasil membaca 8 halaman dari docs_telco_v2.pdf


Setup Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,     
    chunk_overlap=50,   
    separators=["\n\n", "\n", ".", " "]
)

chunked_docs = []
chunk_id = 0

for page in raw_pages:
    chunks = text_splitter.split_text(page["text"])
    for chunk in chunks:
        chunked_docs.append({
            "id": f"chunk_{chunk_id}",
            "text": chunk,
            "metadata": page["metadata"]
        })
        chunk_id += 1

print(f"Total chunks yang dihasilkan: {len(chunked_docs)}")

Total chunks yang dihasilkan: 14


Setup Keyword Search -> BM25

In [ ]:
corpus_texts = [doc["text"] for doc in chunked_docs]

tokenized_corpus = [text.lower().split() for text in corpus_texts]

bm25 = BM25Okapi(tokenized_corpus)

def search_bm25(query, top_k=5):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        if scores[idx] > 0:
            results.append({
                "id": chunked_docs[idx]["id"],
                "text": chunked_docs[idx]["text"],
                "metadata": chunked_docs[idx]["metadata"],
                "score": scores[idx]
            })
    return results

print("BM25 Model berhasil diinisialisasi.")

BM25 Model berhasil diinisialisasi.


Setup Embedding Model -> huggingface:paraphrase-multilingual-MiniLM-L12-v2

In [ ]:
# model billingual
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("Generating embeddings...")
corpus_embeddings = embedding_model.encode(corpus_texts, show_progress_bar=True)

print(f"Shape dari embeddings: {corpus_embeddings.shape}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3533.92it/s]


Generating embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.42it/s]

Shape dari embeddings: (14, 384)


Setup ChromaDB

In [ ]:
chroma_client = chromadb.PersistentClient(path="./chroma_telco")

try:
    chroma_client.delete_collection(name="telco_collection")
except:
    pass

chroma_collection = chroma_client.create_collection(
    name="telco_collection",
    metadata={"hnsw:space": "cosine"}
)

ids = [doc["id"] for doc in chunked_docs]
metadatas = [{"page": str(doc["metadata"]["page"]), "source": doc["metadata"]["source"]} for doc in chunked_docs]

# load data ke ChromaDB
chroma_collection.add(
    embeddings=corpus_embeddings.tolist(),
    documents=corpus_texts,
    metadatas=metadatas,
    ids=ids
)

print(f"Berhasil mengunggah {len(chunked_docs)} chunks ke ChromaDB lokal.")

Berhasil mengunggah 14 chunks ke ChromaDB lokal.


Setup Pinecone

In [ ]:
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
pinecone_index = None

if PINECONE_API_KEY:
    try:
        pc = Pinecone(api_key=PINECONE_API_KEY)
        index_name = "telco-search"
        
        if index_name not in pc.list_indexes().names():
            print(f"Membuat index Pinecone '{index_name}'...")
            pc.create_index(
                name=index_name,
                dimension=384,
                metric='cosine',
                spec=ServerlessSpec(cloud='aws', region='us-east-1')
            )
            
        pinecone_index = pc.Index(index_name)
        
        vectors = []
        for i, doc in enumerate(chunked_docs):
            vectors.append({
                "id": doc["id"],
                "values": corpus_embeddings[i].tolist(),
                "metadata": {
                    "text": doc["text"],
                    "page": str(doc["metadata"]["page"]),
                    "source": doc["metadata"]["source"]
                }
            })
            
        pinecone_index.upsert(vectors=vectors)
        print(f"Berhasil mengunggah {len(vectors)} chunks ke Pinecone Cloud.")
        
    except Exception as e:
        print(f"Error pada Pinecone: {e}")
else:
    print("PINECONE_API_KEY tidak ditemukan di .env. Melewati setup Pinecone.")

Membuat index Pinecone 'telco-search'...
Berhasil mengunggah 14 chunks ke Pinecone Cloud.


Setup Hybrid Search -> RRF

In [ ]:
def reciprocal_rank_fusion(bm25_results, vector_results, k=60):
    """Menggabungkan hasil Keyword dan Semantic menggunakan skor RRF"""
    rrf_scores = {}
    
    # Skor untuk BM25
    for rank, item in enumerate(bm25_results):
        doc_id = item['id']
        if doc_id not in rrf_scores:
            rrf_scores[doc_id] = {'score': 0, 'data': item}
        rrf_scores[doc_id]['score'] += 1 / (k + rank + 1)
        
    # Skor untuk Vector Search
    for rank, item in enumerate(vector_results):
        doc_id = item['id']
        if doc_id not in rrf_scores:
            rrf_scores[doc_id] = {'score': 0, 'data': item}
        rrf_scores[doc_id]['score'] += 1 / (k + rank + 1)
        
    # Urutan berdasarkan skor RRF tertinggi
    sorted_results = sorted(rrf_scores.values(), key=lambda x: x['score'], reverse=True)
    return [item['data'] for item in sorted_results]


def hybrid_search_chroma(query, top_k=3):
    bm25_res = search_bm25(query, top_k=top_k)
    
    query_embed = embedding_model.encode([query]).tolist()
    semantic_res_raw = chroma_collection.query(query_embeddings=query_embed, n_results=top_k)
    
    semantic_res = []
    if semantic_res_raw['ids'][0]:
        for i in range(len(semantic_res_raw['ids'][0])):
            semantic_res.append({
                "id": semantic_res_raw['ids'][0][i],
                "text": semantic_res_raw['documents'][0][i],
                "metadata": semantic_res_raw['metadatas'][0][i]
            })
            
    # 3. Gabungkan dengan RRF
    final_results = reciprocal_rank_fusion(bm25_res, semantic_res)
    return final_results[:top_k]


def hybrid_search_pinecone(query, top_k=3):
    if not pinecone_index:
        return [{"text": "Pinecone tidak tersedia. Harap set API Key."}]
        
    bm25_res = search_bm25(query, top_k=top_k)
    
    query_embed = embedding_model.encode([query]).tolist()[0]
    semantic_res_raw = pinecone_index.query(vector=query_embed, top_k=top_k, include_metadata=True)
    
    semantic_res = []
    for match in semantic_res_raw['matches']:
        semantic_res.append({
            "id": match['id'],
            "text": match['metadata']['text'],
            "metadata": {"page": match['metadata']['page'], "source": match['metadata']['source']}
        })
        
    final_results = reciprocal_rank_fusion(bm25_res, semantic_res)
    return final_results[:top_k]

print("Fungsi Hybrid Search (RRF) berhasil didefinisikan.")

Fungsi Hybrid Search (RRF) berhasil didefinisikan.


test ChromaDB Hybrid Search.

In [10]:
test_queries = [
    "Berapa jumlah BTS di PT NTD.",
    "Berikan Langkah registrasi prabayar.",
    "Layanan apa saja yang ada di PT NTD.",
    "Berikan daftar paket Internet Paling Murah."
]

print("="*60)
print("HASIL HYBRID SEARCH (BM25 + CHROMADB)")
print("="*60)

for idx, query in enumerate(test_queries, 1):
    print(f"\n[Query {idx}] : {query}")
    results = hybrid_search_chroma(query, top_k=2)
    
    for i, res in enumerate(results, 1):
        page = res.get('metadata', {}).get('page', 'Unknown')
        print(f"  ➜ Hasil {i} (Hal {page}): {res['text']}")
    print("-" * 60)

HASIL HYBRID SEARCH (BM25 + CHROMADB)

[Query 1] : Berapa jumlah BTS di PT NTD.
  ➜ Hasil 1 (Hal 1): PT Nusantara Telekomunikasi Digital (NTD) 1. Profil Perusahaan PT Nusantara Telekomunikasi Digital (NTD) adalah perusahaan telekomunikasi nasional yang berdiri pada tahun 2008 dan berfokus pada penyediaan layanan jaringan seluler, internet broadband, serta solusi digital enterprise. Perusahaan ini beroperasi di lebih dari 150 kota di Indonesia dan memiliki lebih dari 35 juta pelanggan aktif
  ➜ Hasil 2 (Hal 4): ● Managed Security Services ● Data Center Colocation Kontribusi Pendapatan Segmen Kontribusi Pendapatan Retail 65% Enterprise 35% Data center utama berada di Jakarta dan Surabaya dengan sertifikasi Tier III. 4. Infrastruktur Jaringan NTD memiliki: ● 45.000 BTS aktif ● 120.000 km fiber optic backbone ● 3 pusat Network Operation Center (NOC) Ringkasan Infrastruktur Infrastruktur Jumlah BTS 45
------------------------------------------------------------

[Query 2] : Berikan Langkah 

test pinecone Hybrid Search.

In [11]:
print("="*60)
print("HASIL HYBRID SEARCH (BM25 + PINECONE)")
print("="*60)

if pinecone_index:
    for idx, query in enumerate(test_queries, 1):
        print(f"\n[Query {idx}] : {query}")
        results = hybrid_search_pinecone(query, top_k=2)
        
        for i, res in enumerate(results, 1):
            page = res.get('metadata', {}).get('page', 'Unknown')
            print(f"  ➜ Hasil {i} (Hal {page}): {res.get('text', res)}")
        print("-" * 60)
else:
    print("Pinecone tidak terkonfigurasi. Silakan periksa .env Anda.")

HASIL HYBRID SEARCH (BM25 + PINECONE)

[Query 1] : Berapa jumlah BTS di PT NTD.
  ➜ Hasil 1 (Hal 1): PT Nusantara Telekomunikasi Digital (NTD) 1. Profil Perusahaan PT Nusantara Telekomunikasi Digital (NTD) adalah perusahaan telekomunikasi nasional yang berdiri pada tahun 2008 dan berfokus pada penyediaan layanan jaringan seluler, internet broadband, serta solusi digital enterprise. Perusahaan ini beroperasi di lebih dari 150 kota di Indonesia dan memiliki lebih dari 35 juta pelanggan aktif
  ➜ Hasil 2 (Hal 4): ● Managed Security Services ● Data Center Colocation Kontribusi Pendapatan Segmen Kontribusi Pendapatan Retail 65% Enterprise 35% Data center utama berada di Jakarta dan Surabaya dengan sertifikasi Tier III. 4. Infrastruktur Jaringan NTD memiliki: ● 45.000 BTS aktif ● 120.000 km fiber optic backbone ● 3 pusat Network Operation Center (NOC) Ringkasan Infrastruktur Infrastruktur Jumlah BTS 45
------------------------------------------------------------

[Query 2] : Berikan Langkah 